# 因子框架快速入门

通过“准备小数据 → 定义衍生因子 → 读取结果 → 批量计算 → 持久化”完成一次完整流程。示例只使用本篇创建的数据和临时目录，无需外部数据库。对真实数据源，用 `getTable().getFactor()` 替换基础因子即可，连接方式见 [JYDB](JYDB.ipynb)、[SQLDB](SQLDB.ipynb) 和 [BaoStockDB](BaoStockDB.ipynb)。

In [ ]:
import datetime as dt
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

from QuantStudio.Core.QSObject import Panel
from QuantStudio.Factor.HDF5DB import HDF5DB
from QuantStudio.Tools.Visualization import qs_help

## 准备基础数据

`FactorDB → FactorTable → Factor` 分别对应库、表和因子。下面创建一个仅有五个日期、两只证券的 HDF5 演示库；`TemporaryDirectory` 管理示例目录，真实应用将其替换为自己的数据目录。

数据组织和接口概览见 [基本框架](基本框架.ipynb)。

In [ ]:
demo_directory = TemporaryDirectory()
HDB = HDF5DB(args={"MainDir": demo_directory.name}).connect()
DTRuler = list(pd.date_range("2025-01-01", periods=5).to_pydatetime())
IDs = ["A", "B"]
prices = pd.DataFrame([[10., 20.], [11., 19.], [12., 21.], [13., 22.], [14., 23.]],
                      index=DTRuler, columns=IDs)
HDB.writeData(Panel({"close": prices, "high": prices + 1, "low": prices - 1}),
              table_name="行情")
print(HDB.TableNames)

# 2. 获取基础因子

通过 `getTable` → `getFactor` 获取基础因子。基础因子是计算图的叶子节点，数据来源于底层数据库。

In [ ]:
# 从 HDF5DB 获取因子
FT = HDB.getTable("行情")
Close = FT.getFactor("close")
High, Low = FT.getFactor("high"), FT.getFactor("low")

print(f"因子名称: {Close.Name}")
print(f"数据类型: {Close.getMetaData(key='DataType')}")

# 3. 读取因子数据

调用 `readData(ids, dts)` 即可读取因子的时序-截面数据，返回 `DataFrame`。

> 因子表也可以直接 `readData`，返回的 `Panel` 是 DataFrames 的集合。详见 **[基本框架](基本框架.ipynb)**。

In [ ]:
DTs = DTRuler[-3:]
print(Close.readData(ids=IDs, dts=DTs))

# 4. 定义衍生因子

衍生因子由**算子（`FactorOperator`）**作用于描述子（依赖因子）产生。QuantStudio 支持四种运算类型：

| 运算 | 算子类 | 典型场景 |
|------|--------|----------|
| 单点运算 | `PointOperator` | 估值指标：PB = 总市值 / 股东权益 |
| 时序运算 | `TimeOperator` | 移动平均线、EMA |
| 截面运算 | `SectionOperator` | 标准化、截面排名 |
| 面板运算 | `PanelOperator` | 双重标准化 |

详细说明请参见 **[因子开发](因子开发.ipynb)**。

## 4.1 表达式方式（单点运算）

最简单的因子定义方式，直接使用 Python 运算符。本质上是 `BasicOperator` 提供的单点运算快捷方式。

In [ ]:
from QuantStudio.Factor.BasicOperator import rename

Mid = rename((High + Low) / 2, factor_name="Mid")
np.testing.assert_allclose(Mid.readData(ids=IDs, dts=DTs), prices.loc[DTs])
print(Mid.readData(ids=IDs, dts=DTs))

## 自定义时序算子

移动平均需要提供完整的 `dt_ruler`，让时序算子能读取请求范围之前的历史数据。三期均线的 `LookBack` 为 2（不包括当前期）。模型参数使用 `OperatorModelArgs`，公开输入使用 `FactorInputModel` 声明。

In [ ]:
from pydantic import Field
from QuantStudio.Factor.Factor import Factor
from QuantStudio.Factor.FactorOperation import (
    makeFactorOperator, OperatorModelArgs, FactorInputModel,
)

class MAArgs(OperatorModelArgs):
    window: int = Field(default=3, ge=1, title="窗口")

class SingleInput(FactorInputModel):
    value: Factor = Field(title="输入因子")

def moving_average(f, idt, iid, x, args: MAArgs):
    """计算给定窗口的移动平均。"""
    return pd.DataFrame(x[0]).rolling(args.window).mean().values[args.window - 1:]

calcMA = makeFactorOperator(
    moving_average, operator_type="Time", model_args=MAArgs, input_model=SingleInput,
    args={"DTMode": "多时点", "IDMode": "多ID", "LookBack": [2]},
)
MA3 = calcMA(value=Close, factor_args={"Name": "MA3"})
result = MA3.readData(ids=IDs, dts=DTs, dt_ruler=DTRuler)
np.testing.assert_allclose(result, prices.rolling(3).mean().loc[DTs])
print(result)

## 4.3 装饰器方式（截面运算）

使用 `@FactorOperatorized` 装饰器是更简洁的算子定义方式。截面运算处理同时点全部证券的数据。

In [ ]:
from QuantStudio.Factor.FactorOperation import FactorOperatorized

class ZScoreArgs(OperatorModelArgs):
    ddof: int = Field(default=0, ge=0, title="自由度")

@FactorOperatorized(operator_type="Section", args={"DTMode": "多时点"},
                   model_args=ZScoreArgs, input_model=SingleInput)
def calcZScore(f, idt, iid, x, args: ZScoreArgs):
    """按每期完整证券截面标准化。"""
    values = x[0]
    return ((values - np.nanmean(values, axis=1, keepdims=True)) /
            np.nanstd(values, axis=1, keepdims=True, ddof=args.ddof))

Close_ZScore = calcZScore(value=Close, factor_args={"Name": "Close_ZScore"})
print(Close_ZScore.readData(ids=IDs, dts=DTs, section_ids=IDs))

## 4.4 使用内置算子

QuantStudio 预定义了 30+ 个常用算子，可直接使用。完整列表请参见 **[因子开发](因子开发.ipynb#builtin-operators)**。

In [ ]:
from QuantStudio.Factor.FactorOperator import Log

# 取对数
LogClose = Log(base=np.e)(Close, factor_args={"Name": "LogClose"})

print("对数收盘价:")
print(LogClose.readData(ids=IDs, dts=DTs))

# 5. 用计算图引擎执行

当需要批量计算多个因子并利用缓存时，可以将因子集合交给引擎执行。引擎负责遍历 DAG、调度节点、管理缓存。

详细说明请参见 **[计算图框架](../Core/计算图框架.ipynb)** 和 **[计算引擎](../Core/计算引擎.ipynb)**。

In [ ]:
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Core.CalcEngine import Engine

# 定义一组因子
Factors = [Close, Mid, MA3, Close_ZScore]

# 创建上下文和引擎
SectionIDs = Close.getID()
Context = FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs)
ExecEngine = Engine()

# 执行计算
from QuantStudio.Factor.Factor import FactorLocalContext

Rslt = ExecEngine.run(
    Factors, Context,
    fwd_data_list=[FactorLocalContext(DTs=DTs, IDs=IDs, SectionIDs=SectionIDs)] * len(Factors)
)

# 查看结果
for i, iFactor in enumerate(Factors):
    print(f"{iFactor.Name}:")
    print(Rslt[i])
    print()

# 6. 写入 HDF5DB 持久化

计算好的衍生因子可以通过 `writeData` 写入 HDF5DB 持久化保存。也可以使用 `FactorStorer` 节点在引擎执行时自动写入。详见 **[HDF5DB](HDF5DB.ipynb)**。

In [ ]:
from QuantStudio.Core.QSObject import Panel

# 将引擎计算得到的结果组装成 Panel 写入
Data = Panel({Factors[i].Name: Rslt[i] for i in range(len(Factors))})
HDB.writeData(data=Data, table_name="my_factors", if_exists="update", data_type={f.Name: f.getMetaData(key="DataType") for f in Factors})

print(f"写入后的表列表: {HDB.TableNames}")
print(f"my_factors 因子列表: {HDB.getTable('my_factors').FactorNames}")

In [ ]:
# 释放连接并移除本篇创建的临时数据。
HDB.disconnect()
demo_directory.cleanup()

# 下一步

| 想了解什么 | 推荐文档 |
|------------|----------|
| 因子框架的架构和 API 总览 | **[基本框架](基本框架.ipynb)** |
| 四种运算的详细参数和用法 | **[因子开发](因子开发.ipynb)** |
| 聚源数据库的使用 | **[JYDB](JYDB.ipynb)** |
| 本地 HDF5 文件存储 | **[HDF5DB](HDF5DB.ipynb)** |
| 计算图原理和引擎调度 | **[计算图框架](../Core/计算图框架.ipynb)** |
| 不同引擎的并行策略 | **[计算引擎](../Core/计算引擎.ipynb)** |